In [ ]:
%pip install numpy pandas scipy scikit-learn

Note: you may need to restart the kernel to use updated packages.


ERROR: Could not find a version that satisfies the requirement system (from versions: none)
ERROR: No matching distribution found for system


In [2]:
import random
import re
from pathlib import Path
from collections import Counter
import sys


sys.path.append("..")
from src.load_data import load
from src.preprocessors import normalize_text

import numpy as np
import pandas as pd
from scipy import sparse
from sklearn.base import clone
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split


# Add imports needed by your chosen NLP techniques in your own cells.


Basicamente la celda de abajo es casi identica a la de la actividad 8.2, solamente se carga todo el dataset, antes se cargaba con load_parrtition por que eran puros .txt

In [3]:
activity_root = Path.cwd()
if activity_root.name == "notebooks":
    activity_root = activity_root.parent
dataset_root = activity_root / "data" / "SMSSpamCollection"


def load_dataset(path: Path) -> pd.DataFrame:
    if not path.is_file():
        raise FileNotFoundError(
           f"Expected 12,500 reviews in {directory}. "
                "Download and extract the complete IMDb dataset; see README.md."
        )
    df = pd.read_csv(path, sep="\t", header=None, names=["label", "text"], encoding="utf-8")
    df["label"] = (df["label"] == "spam").astype(int)
    return df


messages = load_dataset(dataset_root)

train_val, test_reviews = train_test_split(
    messages, test_size=0.15, random_state=42, stratify=messages["label"]
)
train_reviews, val_reviews = train_test_split(
    train_val, test_size=0.15 / 0.85, random_state=42, stratify=train_val["label"]
)
X_train = train_reviews["text"].copy()
X_val = val_reviews["text"].copy()
X_test = test_reviews["text"].copy()
y_train = train_reviews["label"].copy()
y_val = val_reviews["label"].copy()
y_test = test_reviews["label"].copy()
_fixed_inputs = tuple(series.copy(deep=True) for series in (X_train, X_val, X_test, y_train, y_val, y_test))
print("Training reviews:", len(train_reviews), "Validation reviews:", len(val_reviews), "Test reviews:", len(test_reviews))
print("Training labels:", y_train.value_counts().sort_index().to_dict())
print("Validation labels:", y_val.value_counts().sort_index().to_dict())
print("Test labels:", y_test.value_counts().sort_index().to_dict())

Training reviews: 3900 Validation reviews: 836 Test reviews: 836
Training labels: {0: 3377, 1: 523}
Validation labels: {0: 724, 1: 112}
Test labels: {0: 724, 1: 112}


In [6]:
def _check_fixed_inputs():
    for actual, expected in zip((X_train, X_test, y_train, y_test), _fixed_inputs):
        pd.testing.assert_series_equal(actual, expected)


def _check_classifier(pipeline):
    assert isinstance(pipeline, Pipeline), "Create a sklearn Pipeline instance."
    assert "vectorizer" in pipeline.named_steps
    assert pipeline.steps[-1][0] == "classifier"
    classifier = pipeline.named_steps["classifier"]
    assert isinstance(classifier, LogisticRegression)
    assert classifier.get_params() == LogisticRegression(max_iter=1000).get_params()
    vectorizer = pipeline.named_steps["vectorizer"]
    assert isinstance(vectorizer, (CountVectorizer, TfidfVectorizer))
    return vectorizer, classifier


def _check_fitted_pipeline(pipeline, predictions, accuracy):
    _check_fixed_inputs()
    vectorizer, classifier = _check_classifier(pipeline)
    assert isinstance(predictions, np.ndarray)
    assert predictions.shape == (len(X_test),)
    assert set(predictions.tolist()).issubset({0, 1})
    np.testing.assert_array_equal(predictions, pipeline.predict(X_test))
    assert np.isscalar(accuracy) and 0.0 <= accuracy <= 1.0
    np.testing.assert_allclose(accuracy, accuracy_score(y_test, predictions), rtol=0, atol=1e-12)
    matrix = pipeline[:-1].transform(X_test)
    assert sparse.issparse(matrix), "Keep document-term matrices sparse."
    assert matrix.shape[0] == len(X_test)
    np.testing.assert_array_equal(classifier.classes_, [0, 1])

    # Teacher check: refit the declared design on training data only.
    reference = clone(pipeline)
    reference.fit(_fixed_inputs[0], _fixed_inputs[2])
    reference_vectorizer = reference.named_steps["vectorizer"]
    reference_classifier = reference.named_steps["classifier"]
    assert vectorizer.vocabulary_ == reference_vectorizer.vocabulary_
    if isinstance(vectorizer, TfidfVectorizer) and vectorizer.use_idf:
        np.testing.assert_allclose(vectorizer.idf_, reference_vectorizer.idf_)
    np.testing.assert_allclose(classifier.coef_, reference_classifier.coef_, rtol=1e-5, atol=1e-7)
    np.testing.assert_allclose(classifier.intercept_, reference_classifier.intercept_, rtol=1e-5, atol=1e-7)


def _check_independent(pipeline, others):
    for other in others:
        assert pipeline is not other, "Construct a new Pipeline instance."
        for step in ("vectorizer", "classifier"):
            assert pipeline.named_steps[step] is not other.named_steps[step], (
                f"Each pipeline needs its own new {step} instance."
            )
        assert (
            [(type(step), step.get_params(deep=False)) for _, step in pipeline.steps[:-1]]
            != [(type(step), step.get_params(deep=False)) for _, step in other.steps[:-1]]
        ), "Change preprocessing or vectorization, not just the variable name."


In [7]:
import sys
sys.path.append(str(activity_root))   # para que el notebook encuentre la carpeta src/
from src.preprocessors import normalize_text

In [8]:
baseline_pipeline = Pipeline([
    ("vectorizer", CountVectorizer(preprocessor=normalize_text)),
    ("classifier", LogisticRegression(max_iter=1000)),
])
_supplied_baseline = baseline_pipeline


In [9]:
baseline_pipeline.fit(X_train, y_train)
baseline_predictions = baseline_pipeline.predict(X_val)
baseline_accuracy = accuracy_score(y_val, baseline_predictions)

In [10]:
alternative_1_pipeline = Pipeline([
    ("vectorizer",TfidfVectorizer(preprocessor=normalize_text)),
    ("classifier", LogisticRegression(max_iter=1000)),
])
alternative_1_pipeline.fit(X_train, y_train)
alternative_1_predictions = alternative_1_pipeline.predict(X_val)
alternative_1_accuracy = accuracy_score(y_val, alternative_1_predictions)


In [11]:
alternative_2_pipeline = Pipeline([
    ("vectorizer", CountVectorizer(preprocessor=normalize_text, ngram_range=(1, 2))),
    ("classifier", LogisticRegression(max_iter=1000)),
])
alternative_2_pipeline.fit(X_train, y_train)
alternative_2_predictions = alternative_2_pipeline.predict(X_val)
alternative_2_accuracy = accuracy_score(y_val, alternative_2_predictions)

In [12]:
results = pd.DataFrame({
    "pipeline": ["baseline", "alternative_1", "alternative_2"],
    "accuracy": [baseline_accuracy, alternative_1_accuracy, alternative_2_accuracy],
})
display(results.style.format({"accuracy": "{:.2%}"}))

pipelines = {
    "baseline": baseline_pipeline,
    "alternative_1": alternative_1_pipeline,
    "alternative_2": alternative_2_pipeline,
}

names = ["baseline", "alternative_1", "alternative_2"]
pipelines = [baseline_pipeline, alternative_1_pipeline, alternative_2_pipeline]
accuracies = [baseline_accuracy, alternative_1_accuracy, alternative_2_accuracy]

results = pd.DataFrame({"pipeline": names, "accuracy": accuracies})
for name, acc in zip(names, accuracies):
    print(f"{name}: {acc:.2%}")

bestIndex = accuracies.index(max(accuracies))
best_name = names[bestIndex]
best_pipeline = pipelines[bestIndex]

print(f"Best pipeline: {best_name}")

,pipeline,accuracy
0,baseline,98.33%
1,alternative_1,96.29%
2,alternative_2,98.21%


baseline: 98.33%
alternative_1: 96.29%
alternative_2: 98.21%
Best pipeline: baseline


In [13]:
mensajes = [
    "Congratulations! You have won a free prize of 1000 pounds. Call now to claim!",
    "Hey, are we still meeting for lunch tomorrow?",
    "URGENT! Your mobile number has won a 2000 cash award. Text WIN to 80086",
    "Can you send me the notes from class?",
]
print(best_pipeline.predict(mensajes))               # 1 = spam, 0 = legítimo
print(best_pipeline.predict_proba(mensajes)[:, 1])   # probabilidad de spam

[1 0 1 0]
[0.98521666 0.00303609 0.96634128 0.00669974]


In [14]:
from sklearn.dummy import DummyClassifier
from sklearn.metrics import confusion_matrix

majority = DummyClassifier(strategy="most_frequent")
majority.fit(X_train, y_train)
majority_accuracy = accuracy_score(y_test, majority.predict(X_test))

final_predictions = best_pipeline.predict(X_test)
final_accuracy = accuracy_score(y_test, final_predictions)

print(f"{best_name}: {final_accuracy:.2%}")
print(f"Referencia mayoritaria (test): {majority_accuracy:.2%}")
print("Matriz de confusión [[VN, FP], [FN, VP]]:")
print(confusion_matrix(y_test, final_predictions))

errors = pd.DataFrame({"text": X_test, "real": y_test, "pred": final_predictions})
false_positives = errors[(errors["real"] == 0) & (errors["pred"] == 1)]
false_negatives = errors[(errors["real"] == 1) & (errors["pred"] == 0)]
print("Falsos positivos:", len(false_positives), "| Falsos negativos:", len(false_negatives))
display(false_positives)
display(false_negatives)

baseline: 97.61%
Referencia mayoritaria (test): 86.60%
Matriz de confusión [[VN, FP], [FN, VP]]:
[[723   1]
 [ 19  93]]
Falsos positivos: 1 | Falsos negativos: 19


,text,real,pred
5046,We have sent JD for Customer Service cum Accou...,0,1


,text,real,pred
1460,Bought one ringtone and now getting texts cost...,1,0
2823,ROMCAPspam Everyone around should be respondin...,1,0
761,"Romantic Paris. 2 nights, 2 flights from £79 B...",1,0
2804,FreeMsg>FAV XMAS TONES!Reply REAL,1,0
3302,RCT' THNQ Adrian for U text. Rgds Vatian,1,0
731,Email AlertFrom: Jeri StewartSize: 2KBSubject:...,1,0
2863,Adult 18 Content Your video will be with you s...,1,0
1154,1000's of girls many local 2 u who r virgins 2...,1,0
1430,For sale - arsenal dartboard. Good condition b...,1,0
2247,Hi ya babe x u 4goten bout me?' scammers getti...,1,0


In [15]:
train = pd.DataFrame({"text": X_train, "label": y_train})
train["clase"] = train["label"].map({0: "ham", 1: "spam"})

In [17]:
print("Distribución de clases")
print(train["clase"].value_counts())
print((train["clase"].value_counts(normalize=True) * 100).round(1))

Distribución de clases
clase
ham     3377
spam     523
Name: count, dtype: int64
clase
ham     86.6
spam    13.4
Name: proportion, dtype: float64


In [18]:
train["palabras"] = train["text"].str.split().str.len()
print("Longitud de palabras por clase")
display(train.groupby("clase")["palabras"].describe().round(1))

Longitud de palabras por clase


,count,mean,std,min,25%,50%,75%,max
clase,,,,,,,,
ham,3377.0,14.2,11.2,1.0,7.0,11.0,19.0,171.0
spam,523.0,24.0,5.8,3.0,22.0,25.0,28.0,35.0


In [19]:
print("Palabras mas frecuentes por clase")
for clase in ("ham", "spam"):
    palabras = Counter()
    for text in train.loc[train["clase"] == clase, "text"]:
        palabras.update(normalize_text(text).split())
    print(clase, palabras.most_common(15))

Palabras mas frecuentes por clase
ham [('i', 2088), ('you', 1337), ('to', 1073), ('the', 769), ('a', 730), ('u', 714), ('in', 580), ('and', 577), ('my', 540), ('me', 534), ('is', 509), ('it', 491), ('that', 387), ('of', 370), ('for', 350)]
spam [('to', 498), ('a', 283), ('call', 252), ('you', 219), ('your', 198), ('free', 164), ('the', 144), ('now', 141), ('for', 138), ('p', 136), ('or', 129), ('u', 129), ('txt', 127), ('is', 121), ('from', 102)]
